# 11 · Context selection experiments

What evidence would reach Gemini under different context strategies and budgets: expected-document coverage, evidence-term coverage (held-out only), figures for quantity questions, redundancy and source diversity. These are retrieval-side proxies; answer quality is not measured here.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
CTX = {ds: pd.read_csv(A.RESULTS / f"context_{ds}.csv") for ds in ("pilot_v0.1", "heldout_v0.2")
       if (A.RESULTS / f"context_{ds}.csv").exists()}
for ds, c in CTX.items():
    print(ds, len(c), "rows; strategies:", sorted(c["strategy"].unique()), "; budgets:", sorted(c["budget_chars"].unique()))

In [ ]:
def ctx_summary(c):
    c = c.copy()
    for col in ("evidence_terms_found", "figure_in_expected_passage"):
        c[col] = pd.to_numeric(c[col], errors="coerce")
    rows = []
    for (st, b), g in c.groupby(["strategy", "budget_chars"]):
        mean, lo, hi = A.cluster_bootstrap(g, "expected_document_in_context")
        e = A.cluster_bootstrap(g, "evidence_terms_found") if g["evidence_terms_found"].notna().any() else (np.nan,) * 3
        q = g[g.quantity_question == 1]
        rows.append(dict(strategy=st, budget_chars=b, n=len(g), expected_doc_in_context=mean, lo=lo, hi=hi,
                         evidence_terms_found=e[0], evidence_lo=e[1], evidence_hi=e[2],
                         figure_for_quantity_questions=q["figure_in_expected_passage"].mean() if len(q) else np.nan,
                         n_quantity=len(q), redundant_pair_share=g["redundant_pair_share"].mean(),
                         documents=g["documents"].mean(), passages=g["passages"].mean(), chars=g["chars"].mean()))
    return pd.DataFrame(rows)
SUM = {ds: ctx_summary(c) for ds, c in CTX.items()}
pd.concat([s.assign(dataset=ds) for ds, s in SUM.items()]).round(3).to_csv(A.RESULTS / "context_summary.csv", index=False)
SUM["heldout_v0.2"].round(3)

In [ ]:
s = SUM["heldout_v0.2"]
ST = {"top_k_no_rerank": ("Top-K, no reranking", A.COLORS["B0"]), "top_k_reranked": ("Top-K after reranking", A.COLORS["E1"]),
      "document_balanced": ("Document-balanced (≤2 per document)", A.COLORS["E2"]),
      "question_aware_selection": ("Question-aware selection (assistant)", A.COLORS["E2S"])}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
for ax, (col, lo, hi, label) in zip(axes, [("expected_doc_in_context", "lo", "hi", "expected document in context"),
                                           ("evidence_terms_found", "evidence_lo", "evidence_hi", "evidence terms found in context")]):
    for st, (name, color) in ST.items():
        g = s[s.strategy == st].sort_values("budget_chars")
        if g.empty: continue
        ax.errorbar(g["budget_chars"], g[col], yerr=[g[col] - g[lo], g[hi] - g[col]], color=color, marker="o",
                    linewidth=2, markersize=7, capsize=3, label=name)
    ax.set_xlabel("context budget (characters)"); ax.set_ylabel(f"share of questions (95% CI)"); ax.set_ylim(0, 1.05)
    ax.set_title(f"{label.capitalize()} - held-out (n = 100)"); ax.set_xticks([3000, 6000, 9000])
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.12))
fig.tight_layout(); print(A.save(fig, "11_context_strategies"))

In [ ]:
b9 = s[s.budget_chars == 9000].set_index("strategy")
display(Markdown(f"""**Interpretation (held-out, 9,000-character budget).** Expected document in context: """
    + ", ".join(f"{ST[k][0]} {b9.loc[k, 'expected_doc_in_context']:.2f}" for k in ST if k in b9.index)
    + f""". Redundant passage pairs (token Jaccard ≥ 0.5) are rare in every strategy. Larger budgets mostly add
documents rather than completeness of the right one; quantity questions (n = {int(b9['n_quantity'].iloc[0])}) are where
figures in the expected passage matter.

**Not evaluated**: whether these contexts change Gemini's answer correctness or faithfulness - that requires controlled
generation with a working model and human scoring (Phase 7 gate not met). Contradictory-source detection is not
measured automatically. **Limitations**: evidence terms exist only for held-out scenarios and were drafted by an AI
assistant; these are proxies, not answer-quality measures."""))